In [ ]:
import deepchem as dc
import tensorflow as tf
import numpy as np
import pandas as pd
import os
import tempfile
import random
from sklearn.model_selection import train_test_split

seeds = np.random.randint(0, 100000, size=3).tolist()
tasks = ['Labels']
metric = dc.metrics.Metric(dc.metrics.r2_score, mode="regression")
all_metrics = [dc.metrics.r2_score, dc.metrics.mae_score, dc.metrics.rms_score]
for target_name in ['ESOL','bace','FreeSolv','FreeSolv_cal','ro','Lipophilicity', 'qm8']:
    df = pd.read_csv(f'{target_name}/{target_name}.csv')
    df.columns = ['SMILES','Labels']
    df['Labels'] = pd.to_numeric(df['Labels'], errors='coerce')
    df = df.dropna(subset=['SMILES', 'Labels']).reset_index(drop=True)
    noise_portions = [0.1, 0.3, 0.5, 0.7, 0.9]
    sigmas = [1, 2, 3]
    base_path = f'{target_name}/for_noise_effect'
    os.makedirs(base_path, exist_ok=True)
    for sigma in sigmas:
        for seed in seeds:
            print(f"\n===== Sigma {sigma}, Seed {seed} =====")
            np.random.seed(seed)
            tf.random.set_seed(seed)
            random.seed(seed)
            train_val_df, test_df = train_test_split(df, test_size=0.1, random_state=seed)
            train_df, valid_df = train_test_split(train_val_df, test_size=0.1, random_state=seed)
            origin_path = f"{base_path}/origin/sigma{sigma}_seed{seed}"
            os.makedirs(origin_path, exist_ok=True)
            train_df.to_csv(f'{origin_path}/{target_name}_train.csv', index=False)
            valid_df.to_csv(f'{origin_path}/{target_name}_valid.csv', index=False)
            test_df.to_csv(f'{origin_path}/{target_name}_test.csv', index=False)
            max_frac = max(noise_portions)
            train_max_noise_idx = train_df.sample(frac=max_frac, random_state=seed).index
            valid_max_noise_idx = valid_df.sample(frac=max_frac, random_state=seed).index
            for n in noise_portions:
                noise_path = f""
                os.makedirs(noise_path, exist_ok=True)
                k_train = int(n * len(train_df))
                noise_idx_train = train_max_noise_idx[:k_train]
                train_noisy = train_df.copy()
                noise_std = sigma * train_df['Labels'].std()
                train_noisy.loc[noise_idx_train, 'Labels'] += np.random.normal(
                    loc=0, scale=noise_std, size=len(noise_idx_train)
                )
                k_valid = int(n * len(valid_df))
                noise_idx_valid = valid_max_noise_idx[:k_valid]
                valid_noisy = valid_df.copy()
                noise_std = sigma * valid_df['Labels'].std()
                valid_noisy.loc[noise_idx_valid, 'Labels'] += np.random.normal(
                    loc=0, scale=noise_std, size=len(noise_idx_valid)
                )
                test_noisy = test_df.copy()
                train_noisy.to_csv(f'{noise_path}/{target_name}_train_{n}_{sigma}_{seed}.csv', index=False)
                valid_noisy.to_csv(f'{noise_path}/{target_name}_valid_{n}_{sigma}_{seed}.csv', index=False)
                test_noisy.to_csv(f'{noise_path}/{target_name}_test_{n}_{sigma}_{seed}.csv', index=False)